In [1]:
import regex as re
from collections import defaultdict, Counter
import multiprocessing as mp
import os
from typing import BinaryIO


In [12]:
def split_by_special(
    text: str, 
    special_tokens: list[str], 
    drop_special=True
) -> list[str]:
    if not special_tokens:
        return [text]

    special_tokens = sorted(special_tokens, key=len, reverse=True)
    pattern: str = "|".join(re.escape(tok) for tok in special_tokens)
    
    if not drop_special:
        pattern = f"({pattern})"

    print(f"Split pattern: {pattern}")

    text_chunks: list[str] = re.split(pattern, text)
    return [text_chunk for text_chunk in text_chunks if text_chunk]

In [7]:
text = "Lucy knew that even if others ignore her friend, the spirit was real and they could play together.<|endoftext|>"
special_tokens = [ "<|pad|>", "<|end|>", "<|endoftext|>"]

In [13]:
text_chunks = split_by_special(text, special_tokens, drop_special=False)
print(text_chunks)

Split pattern: (<\|endoftext\|>|<\|pad\|>|<\|end\|>)
['Lucy knew that even if others ignore her friend, the spirit was real and they could play together.', '<|endoftext|>']


In [11]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

words: list[bytes] = []
word2count: Counter[bytes] = Counter()

for text_chunk in text_chunks:
    partial_words = re.findall(PAT, text_chunk)

    for word in partial_words:
        word = word.encode("utf-8")
        word2count[word] += 1

        if word2count[word] == 1:
            words.append(word)

print(words)
print(word2count)

[b'Hello', b'world', b'!', b'This', b' is', b' a', b' test', b'.']
Counter({b'Hello': 1, b'world': 1, b'!': 1, b'This': 1, b' is': 1, b' a': 1, b' test': 1, b'.': 1})


In [3]:
import torch

batch_size = 1
seq_len = 2
hidden_size = 4
num_heads = 2
d_k = hidden_size // num_heads
x = torch.tensor([[[ 0.3778, -1.6829, -0.0558, -1.0177],
         [-0.7682,  1.0619, -0.1749, -0.3386]]])

# split head
x = x.view(batch_size, seq_len, num_heads, d_k)
print(x)
x = x.permute(0, 2, 1, 3)
print(x)

# merge head
x = x.permute(0, 2, 1, 3)
print(x)
x = x.view(batch_size, seq_len, hidden_size)
print(x)

tensor([[[[ 0.3778, -1.6829],
          [-0.0558, -1.0177]],

         [[-0.7682,  1.0619],
          [-0.1749, -0.3386]]]])
tensor([[[[ 0.3778, -1.6829],
          [-0.7682,  1.0619]],

         [[-0.0558, -1.0177],
          [-0.1749, -0.3386]]]])
tensor([[[[ 0.3778, -1.6829],
          [-0.0558, -1.0177]],

         [[-0.7682,  1.0619],
          [-0.1749, -0.3386]]]])
tensor([[[ 0.3778, -1.6829, -0.0558, -1.0177],
         [-0.7682,  1.0619, -0.1749, -0.3386]]])
